# 02 - Fase 3 (preparacion): entrenar Chemprop (DG de solvatacion)

Modelo de **2 componentes**: `(solvente, soluto) -> DG_solv [kcal/mol]`. Se entrena una vez; el nb 03 lo usa en cada ronda.

**Elige la fuente de datos abajo (`DATA_SOURCE`):**
- `"freesolv"` (por defecto, cero setup): baja FreeSolv y arma un CSV de arranque. **Solo agua como solvente** -> sirve para verificar que todo el pipeline corre, pero el modelo NO aprende a distinguir solventes. Dataset de cableado, no de descubrimiento.
- `"zenodo"` (recomendado para ciencia real): baja **CombiSolv-Exp** (Vermeire & Green 2021, ~10k pares, muchos solventes) desde Zenodo. Kaggle tiene internet abierto, asi que se descarga aqui mismo.
- `"upload"`: tu propio CSV subido como Kaggle Dataset (Add Input). Lo mejor si tienes datos de ILs/DES.

Casi ningun dataset publico tiene iones metal-oxo ni muchos ILs/DES; la celda de cobertura lo cuantifica para que sepas cuanto fiarte.

**Settings:** GPU · Internet On · (si `upload`) Add Input -> tu dataset. Save & Run All (Commit).

In [ ]:
# ==== Setup comun: codigo en /tmp (efimero), estado en /kaggle/working/alumina (se guarda) ====
import os, glob, json, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/santiagovelcasallas/Data_Science_example_01.git"
BRANCH   = "claude/affectionate-johnson-q9wtj0"   # cambia a "main" tras el merge

ROOT = Path("/kaggle/working/alumina")
CODE = Path("/tmp/repo")
PKG  = CODE / "alumina_pipeline"
for d in ["data", "models", "priors", "runs"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)
os.environ["ALUMINA_ROOT"] = str(ROOT)

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

token = secret("GITHUB_TOKEN")   # solo si el repo es privado
url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
if not CODE.exists():
    r = subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, url, str(CODE)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git clone fallo: " + r.stderr[-300:])
print("Codigo en", PKG)

In [ ]:
# ==== Instalacion (~3-6 min). Se fija la version de torch de Kaggle para no romper CUDA ====
!pip freeze | grep -E "^torch==" > /tmp/constraints.txt; cat /tmp/constraints.txt
!pip install -q -c /tmp/constraints.txt -r {PKG}/kaggle/requirements-kaggle.txt
!pip install -q --no-deps -e {PKG}
!python -c "import torch,rdkit,ase,botorch,alumina_pipeline as a; print('torch',torch.__version__,'| GPU:',torch.cuda.is_available())"

In [ ]:
DATA_SOURCE = "freesolv"    # "freesolv" | "zenodo" | "upload"
EPOCHS, ENSEMBLE = 100, 3

# Solo para "upload": ajusta a los nombres de columna de TU csv
CSV_PATTERN = "*.csv"
SOLVENT_COL, SOLUTE_COL, TARGET_COL, TARGET_UNITS = "solvent", "solute", "dG", "kcal/mol"

# Solo para "zenodo": record de la coleccion SolProp (contiene CombiSolv-Exp)
ZENODO_RECORD = "5970538"
ZENODO_CSV    = None    # None = auto. Si elige mal, fija aqui parte del nombre del CSV correcto.

In [ ]:
import re, io, zipfile, urllib.request, pandas as pd
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")

def load_freesolv():
    url = "https://raw.githubusercontent.com/MobleyLab/FreeSolv/master/database.txt"
    txt = urllib.request.urlopen(url, timeout=60).read().decode()
    rows = []
    for ln in txt.splitlines():
        if ln.startswith("#") or not ln.strip(): continue
        p = [x.strip() for x in ln.split(";")]
        if len(p) >= 5: rows.append({"solvent": "O", "solute": p[1], "dG": float(p[3])})
    return pd.DataFrame(rows)   # agua (O) como unico solvente

def _find_cols(columns):
    """Devuelve (col_solvente, col_soluto, col_dG) o None si el CSV no sirve."""
    low = {c.lower(): c for c in columns}
    def pick(preds):
        for lc, orig in low.items():
            if all(p(lc) for p in preds): return orig
        return None
    sv = pick([lambda c: "solvent" in c, lambda c: "smile" in c or c == "solvent"]) \
         or pick([lambda c: c == "solvent"])
    su = pick([lambda c: "solute" in c, lambda c: "smile" in c or c == "solute"]) \
         or pick([lambda c: c == "solute"])
    # energia libre de solvatacion (no solubilidad logS, no entalpia Hsolv)
    tg = pick([lambda c: "solv" in c, lambda c: c.startswith(("dg", "g")) and "log" not in c and "h" not in c[:2]]) \
         or pick([lambda c: "dgsolv" in c.replace(" ", "")])
    return (sv, su, tg) if sv and su and tg else None

def load_zenodo(record):
    cache = ROOT / "data" / f"zenodo_{record}.zip"
    if not cache.exists():
        files = json.load(urllib.request.urlopen(f"https://zenodo.org/api/records/{record}", timeout=60))["files"]
        print("Archivos en el record:", [f["key"] for f in files])
        pick = next((f for f in files if f["key"].lower().endswith((".zip", ".csv"))), None)
        print("Descargando (se cachea para proximas corridas):", pick["key"])
        urllib.request.urlretrieve(pick["links"]["self"], cache)
    # inspecciona TODOS los CSVs del zip y elige por columnas, no por nombre
    z = zipfile.ZipFile(cache)
    names = [n for n in z.namelist() if n.lower().endswith(".csv")]
    chosen = None
    print("CSVs dentro del zip:")
    for n in names:
        try:
            cols = list(pd.read_csv(z.open(n), nrows=0).columns)
        except Exception:
            continue
        hit = _find_cols(cols)
        print(f"  {'[USABLE]' if hit else '[  no   ]'} {n}  ->  {cols}")
        if hit and (ZENODO_CSV is None or ZENODO_CSV in n):
            chosen, mapping = n, hit
    if chosen is None:
        raise RuntimeError("Ningun CSV tiene columnas (solvente, soluto, dGsolv). "
                           "Mira la lista de arriba y fija ZENODO_CSV al nombre correcto.")
    sv, su, tg = mapping
    print(f"\nElegido: {chosen}\n  solvente={sv!r}  soluto={su!r}  dG={tg!r}")
    d = pd.read_csv(z.open(chosen))
    return d.rename(columns={sv: "solvent", su: "solute", tg: "dG"})[["solvent", "solute", "dG"]]

def load_upload():
    hits = sorted(glob.glob("/kaggle/input/**/" + CSV_PATTERN + "", recursive=True))
    if not hits: raise FileNotFoundError("Sin CSV: Add Input -> tu dataset")
    d = pd.read_csv(hits[0]).rename(columns={SOLVENT_COL: "solvent", SOLUTE_COL: "solute", TARGET_COL: "dG"})
    if TARGET_UNITS == "kJ/mol": d["dG"] = d["dG"] / 4.184
    return d[["solvent", "solute", "dG"]]

df = {"freesolv": load_freesolv, "zenodo": lambda: load_zenodo(ZENODO_RECORD), "upload": load_upload}[DATA_SOURCE]()
print(f"\nFuente '{DATA_SOURCE}': {len(df)} filas crudas")

In [ ]:
# Limpieza + cobertura del dominio
def canon(s):
    m = Chem.MolFromSmiles(str(s));  return Chem.MolToSmiles(m) if m else None
n0 = len(df)
df = df.dropna(subset=["solvent", "solute", "dG"]).copy()
df["dG"] = pd.to_numeric(df["dG"], errors="coerce")
df["solvent"], df["solute"] = df["solvent"].map(canon), df["solute"].map(canon)
df = df.dropna()
print(f"Filas validas: {len(df)}/{n0} | solventes unicos: {df.solvent.nunique()} | solutos unicos: {df.solute.nunique()}")

ionic = df.solvent.map(lambda s: "." in s or "+" in s).mean()
print(f"Cobertura -> solventes ionicos/mezclas: {ionic:.1%}")
if df.solvent.nunique() == 1:
    print("[!] Un solo solvente: el modelo NO podra distinguir solventes (dataset de cableado).")
elif ionic < 0.01:
    print("[!] Casi no hay solventes ionicos: las predicciones para ILs/DES seran extrapolaciones.")
train_csv = ROOT / "data/solvation_train.csv"; df.to_csv(train_csv, index=False)

In [ ]:
!chemprop train --data-path {train_csv} --smiles-columns solvent solute --target-columns dG \
  --task-type regression --epochs {EPOCHS} --ensemble-size {ENSEMBLE} \
  --output-dir {ROOT}/models/chemprop_solvation 2>&1 | grep -viE "warning|it/s" | tail -n 40

In [ ]:
# Chequeo de cordura: agua en agua ~ -6.3 kcal/mol
test = pd.DataFrame({"solvent":["O","CO","O"], "solute":["O","O","[OH-]"],
                     "nota":["agua/agua ~-6.3","agua/metanol","OH-/agua"]})
test[["solvent","solute"]].to_csv("/tmp/test.csv", index=False)
!chemprop predict --test-path /tmp/test.csv --preds-path /tmp/tp.csv --model-paths {ROOT}/models/chemprop_solvation --smiles-columns solvent solute 2>&1 | tail -n 1
tp = pd.read_csv("/tmp/tp.csv")
test["pred_kcal_mol"] = tp[[c for c in tp.columns if c not in ("solvent","solute")][0]].values
test

## Que guardar

| Archivo | Para que | Imprescindible |
|---|---|---|
| `alumina/models/chemprop_solvation/` (carpeta) | El modelo que usa el nb 03 | **Si** |
| RMSE/MAE de test (impresos arriba) | Cuanto confiar en DG | **Si** (anotalos) |
| `alumina/data/solvation_train.csv` | Reproducibilidad | Recomendado |

En el nb 03: Add Input -> Notebook Output -> este notebook.